# Import Required Libraries

In [1]:
import os
import json
import re

# Read Text Files

In [2]:
def read_text_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


# Example usage
file_path = "/Users/juan/Documents/GTA/Emory_internship/Hypothesis_generation_pipeline/outputs/initial_ouput_biomed/example_run.txt"
content = read_text_file(file_path)
print("File content loaded.")

File content loaded.


# Parse Agent Outputs

In [3]:
import re


def parse_agent_outputs(content):
    # Split by ----------
    sections = re.split(r"^---------- ", content, flags=re.MULTILINE)
    agent_outputs = {}
    for section in sections:
        if not section.strip():
            continue
        lines = section.strip().split("\n")
        first_line = lines[0]
        if "TextMessage" in first_line or "ToolCallSummaryMessage" in first_line:
            agent = first_line.split("(")[1].split(")")[0]
            output = "\n".join(lines[1:]).strip()
            if agent not in agent_outputs:
                agent_outputs[agent] = []
            agent_outputs[agent].append(output)
    # For each agent, take the last output
    final_outputs = {}
    for agent, outputs in agent_outputs.items():
        last_output = outputs[-1]
        if agent == "hypothesis_expansion_agent":
            final_outputs[agent] = last_output
        else:
            try:
                parsed = json.loads(last_output)
                final_outputs[agent] = parsed
            except json.JSONDecodeError:
                final_outputs[agent] = last_output
    return final_outputs


agent_dict = parse_agent_outputs(content)

# Create Dictionary

In [4]:
# The dictionary is created above as agent_dict
print("Agents found:", list(agent_dict.keys()))

Agents found: ['user', 'path_generation_agent', 'ontologist', 'scientist', 'hypothesis_expansion_agent', 'novelty_critic_agent', 'coding_agent_assistant']


# Test the Code

In [5]:
# Test by printing the dictionary
for agent, output in agent_dict.items():
    print(f"Agent: {agent}")
    if isinstance(output, dict):
        print("Output is a dict:")
        for key, value in output.items():
            truncated = str(value)[:100] + "..." if len(str(value)) > 100 else str(value)
            print(f"  {key}: {truncated}")
    else:
        truncated = output[:200] + "..." if len(output) > 200 else output
        print(f"Output: {truncated}")
    print("---")

Agent: user
Output: I would like to explore research ideas between: bladder cancer, male, diabetes.
---
Agent: path_generation_agent
Output: diabetes||Disease -- is associated with; may share -- bladder cancer||Disease -- spreads from -- bladder||Organ -- spreads from -- bladder cancer||Disease -- is one of the top 10 frequently occurring ...
---
Agent: ontologist
Output is a dict:
  definitions: {'diabetes': 'A group of metabolic diseases characterized by chronic hyperglycemia resulting from de...
  relationships: [{'from': 'diabetes', 'to': 'bladder cancer', 'relationship': 'Epidemiological and clinical studies ...
---
Agent: scientist
Output: <thinking>
**Considering gene interactions**

I realize we should focus solely on the ADRM1 gene and specific image features from the list provided. This is okay. The relationships I see suggest ADRM1...
---
Agent: hypothesis_expansion_agent
Output: {
  "notes": "Keep the focus strictly on ADRM1 (RNA expression) and one WSI-derived imaging feat

# Generate Structured Markdown

In [7]:
import json
import re


def humanize_agent_name(agent):
    # Convert snake_case or camelCase to Title Case with spaces
    return " ".join(word.capitalize() for word in agent.replace("_", " ").replace("-", " ").split())


md_content = ""
for agent, output in agent_dict.items():
    human_name = humanize_agent_name(agent)
    md_content += f"# {human_name}\n\n"
    if agent == "scientist":
        # Extract hypothesis from the text
        match = re.search(r'"hypothesis":\s*"([^"]*)"', str(output))
        if match:
            md_content += f"{match.group(1)}\n\n"
        else:
            md_content += f"{output}\n\n"
    elif agent == "hypothesis_expansion_agent":
        if isinstance(output, dict):
            if "expanded_hypothesis" in output:
                md_content += f"**Expanded Hypothesis:** {output['expanded_hypothesis']}\n\n"
            if "justification" in output:
                md_content += f"**Justification:** {output['justification']}\n\n"
        elif isinstance(output, str):
            # Extract expanded_hypothesis
            match_exp = re.search(r'"expanded_hypothesis":\s*"([^"]*(?:\\.[^"]*)*)"', output, re.DOTALL)
            if match_exp:
                md_content += f"**Expanded Hypothesis:** {match_exp.group(1)}\n\n"
            # Extract justification
            match_just = re.search(r'"justification":\s*"([^"]*(?:\\.[^"]*)*)"', output, re.DOTALL)
            if match_just:
                md_content += f"**Justification:** {match_just.group(1)}\n\n"
            if not match_exp and not match_just:
                md_content += f"Error: Could not extract fields\nFirst 100: {output[:100]}\nLast 100: {output[-100:]}\n"
        else:
            md_content += f"{output}\n\n"
    elif agent == "novelty_critic_agent":
        if isinstance(output, dict):
            if "novelty" in output:
                md_content += f"**Novelty:** {output['novelty']}\n\n"
            if "rationale" in output:
                md_content += f"**Rationale:** {output['rationale']}\n\n"
            if "References" in output:
                md_content += f"**References:** {output['References']}\n\n"
        elif isinstance(output, str):
            # Extract novelty
            match_nov = re.search(r'"novelty":\s*(\d+)', output)
            if match_nov:
                md_content += f"**Novelty:** {match_nov.group(1)}\n\n"
            # Extract rationale
            match_rat = re.search(r'"rationale":\s*"([^"]*(?:\\.[^"]*)*)"', output, re.DOTALL)
            if match_rat:
                md_content += f"**Rationale:** {match_rat.group(1)}\n\n"
            # Extract References
            match_ref = re.search(r'"References":\s*\[([^\]]*)\]', output, re.DOTALL)
            if match_ref:
                md_content += f"**References:** [{match_ref.group(1)}]\n\n"
            if not match_nov and not match_rat and not match_ref:
                md_content += f"{output}\n\n"
        else:
            md_content += f"{output}\n\n"
    elif agent == "ontologist":
        if isinstance(output, dict):
            if "definitions" in output:
                md_content += "## Definitions\n\n"
                for term, defn in output["definitions"].items():
                    md_content += f"**{term}:** {defn}\n\n"
            if "relationships" in output:
                md_content += "## Relationships\n\n"
                for rel in output["relationships"]:
                    md_content += f"- **{rel['from']}** to **{rel['to']}:** {rel['relationship']}\n\n"
        else:
            md_content += f"{output}\n\n"
    else:
        if isinstance(output, dict):
            md_content += "```json\n" + json.dumps(output, indent=2) + "\n```\n\n"
        else:
            md_content += f"{output}\n\n"

# Write to file
output_file = "/Users/juan/Documents/GTA/Emory_internship/Hypothesis_generation_pipeline/outputs/initial_ouput_biomed/example_run.md"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(md_content)

print(f"Markdown file updated at {output_file}")

Markdown file updated at /Users/juan/Documents/GTA/Emory_internship/Hypothesis_generation_pipeline/outputs/initial_ouput_biomed/example_run.md
